In [5]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template

In [ ]:
max_seq_length = 1024  # Limited to reduce VRAM
dtype = None           # Auto-detection for the dtype calculation
load_in_4bit = True   # We load in 4-bit

# The model is loaded using Unsloth, which has already applied double quantization NF4 to reduce the model's memory footprint.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.10.4: Fast Mistral patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 5060 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 54.35it/s]


Modèle Mistral-7B chargé avec succès via Unsloth !


In [4]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096, padding_idx=770)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_lay

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                        
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,               # Optimized to 0 for Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth", # To save VRAM
    random_state = 15,
)

Unsloth 2026.10.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Adaptateurs LoRA appliqués !


In [7]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096, padding_idx=770)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

In [ ]:
# Application of the standard Mistral chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "mistral",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) 
        for convo in convos
    ]
    return {"text": texts}

# Load the finetuning dataset
dataset = load_dataset("json", data_files="..\data\processed\patient_dataset.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

Generating train split: 3 examples [00:00, 354.42 examples/s]
Map: 100%|██████████| 3/3 [00:00<00:00, 63.83 examples/s]

Dataset chargé avec succès ! Nombre d'exemples : 3


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 2,
    packing = False, # Désactivé pour préserver la structure des dialogues
    args = SFTConfig(
        per_device_train_batch_size = 2,    # Sécurité VRAM sur 8 Go
        gradient_accumulation_steps = 4,    # Simule un plus gros batch
        warmup_steps = 5,
        max_steps = 100,                    # À ajuster selon la taille de votre dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), # Activé sur votre RTX 5060
        logging_steps = 1,
        output_dir = "../outputs/sft_model",
        optim = "paged_adamw_8bit",          # Économise la VRAM
        seed = 15,
        report_to = "none"
    ),
)

# Lancement du fine-tuning
trainer_stats = trainer.train()
print("Entraînement terminé avec succès !")

Unsloth: We found double BOS tokens - we shall remove one automatically.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.063097


Unsloth: Restored added_tokens_decoder metadata in outputs_mistral_mystere\checkpoint-1\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs_mistral_mystere\checkpoint-1.


Entraînement terminé avec succès !
